# Multi-agent RAG assessment demo

This notebook demonstrates the **Technical Task**, using the same Python package as the API.
Default: real local MiniLM embeddings and an explicit **mock, extractive** LLM.
Set `RAG_MODE=live` and `OPENAI_API_KEY` before launching Jupyter for real OpenAI generation.
`EMBEDDING_BACKEND=lexical` is an explicit download-free test double, not semantic search.

Use **Restart Kernel → Run All**. The first embedding download needs internet.
Each run creates fresh in-memory state; no separate API or database is required.

In [1]:
import json
import os
import sys
from pathlib import Path

root = Path.cwd()
if not (root / "rag_system").exists():
    root = root.parent
assert (root / "rag_system").exists(), "Launch from the repository or notebooks directory"
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from rag_system.bootstrap import build_system
from rag_system.models import Feedback

system = build_system()
print(
    json.dumps(
        {
            "mode": system.llm.mode,
            "embeddings": system.store.embedder.name,
            "documents": len(system.store.documents),
        },
        indent=2,
    )
)

/private/tmp/rag-assessment-venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "mode": "mock",
  "embeddings": "fastembed:sentence-transformers/all-MiniLM-L6-v2",
  "documents": 18
}


## 1. Inspect actual retrieval
Each short synthetic passage is one chunk. Inspect source IDs and raw cosine similarity.
Similarity measures relevance; confidence combines relevance, source authority and feedback.
Neither is a calibrated probability that a claim is true.

In [2]:
hits = system.store.search("deploy microservice", "technical", top_k=3, min_similarity=0)
for hit in hits:
    print(hit.document.source_id, round(hit.similarity, 3), hit.document.title)
assert hits and all(h.document.domain == "technical" for h in hits)

tech-deploy@v1#0 0.696 Deploying a microservice
tech-rollback@v1#0 0.331 Rollback procedure
tech-capacity@v1#0 0.133 Capacity changes


## 2. Required multi-domain scenarios
The planner creates domain-specific tasks. Domain and complexity control retrieval budgets.
Agents return claims with source IDs; the orchestrator resolves conflicts and synthesizes.
The mock planner uses explicit lexical rules; the live planner uses structured LLM output.

In [3]:
queries = [
    "What’s the process for deploying a new microservice and what compliance checks are needed?",
    "How do I troubleshoot API performance issues while following our security policies?",
    "What business approvals are required for implementing a new data processing workflow?",
]
answers = [system.query(query) for query in queries]
for query, answer in zip(queries, answers):
    print("QUESTION:", query, "\nSTATUS:", answer.status)
    print("PLAN:", answer.plan.model_dump())
    print("RETRIEVAL:", [(r.task.domain, r.top_k, len(r.retrieved)) for r in answer.results])
    print(answer.answer, "\n")
    assert answer.status == "answered" and answer.citations

QUESTION: What’s the process for deploying a new microservice and what compliance checks are needed? 
STATUS: answered
PLAN: {'tasks': [{'domain': 'technical', 'subquery': 'What implementation or operational steps apply to this request: What’s the process for deploying a new microservice and what compliance checks are needed?'}, {'domain': 'compliance', 'subquery': 'Which security, privacy or compliance requirements apply to this request: What’s the process for deploying a new microservice and what compliance checks are needed?'}], 'complexity': 'complex', 'reason': 'Mock lexical routing using documented domain signals.'}
RETRIEVAL: [('technical', 4, 4), ('compliance', 5, 5)]
To deploy a microservice, run unit and integration tests, build a versioned container image, deploy to staging, then use a canary rollout with a rollback plan. [tech-deploy@v1#0]

Implement a data processing workflow with schema validation, idempotent processing, retries and a dead-letter queue. Test it in staging

## 3. Resolve contradictory sources
An informal engineering note says **90 days**; the approved compliance policy says **30 days**.
We compare annotated facts with identical scope. Same-scope peers are checked beyond top-k,
so retrieval order cannot hide the policy. This is not arbitrary natural-language conflict detection.

In [4]:
conflict_answer = system.query("How long should we retain production API request logs?")
print(conflict_answer.answer)
for conflict in conflict_answer.conflicts:
    print(conflict.model_dump())
assert any(c.selected_source_id == "comp-retention@v1#0" for c in conflict_answer.conflicts)
assert "90 days" not in conflict_answer.answer

Production API request logs must be deleted after 30 days under the current approved retention policy. This policy takes precedence over informal engineering notes. [comp-retention@v1#0]

Use load tests and CPU and memory measurements to decide whether API capacity needs scaling. Define a rollback threshold before changes. [tech-capacity@v1#0]

To troubleshoot API performance issues, inspect p95 latency, error rates and traces. Profile slow endpoints and database queries before changing capacity. [tech-performance@v1#0]

To deploy a microservice, run unit and integration tests, build a versioned container image, deploy to staging, then use a canary rollout with a rollback plan. [tech-deploy@v1#0]

When troubleshooting API performance, redact personal data and credentials from logs, use least-privilege access and never disable production audit logging. [comp-debug@v1#0]

Before deploying a microservice, complete dependency vulnerability scanning, secrets scanning and a security review. 

## 4. Update knowledge and preserve historical citations
Updates require a higher version and replace the existing embedding. Idempotent retries are safe.
A previous answer retains its original document snapshot. All state resets with a new system.

In [5]:
old = system.store.documents["comp-retention"]
updated = old.model_copy(
    update={"version": 2, "value": "14", "text": old.text.replace("30 days", "14 days")}
)
assert system.upsert(updated)
assert not system.upsert(updated)
after_update = system.query("How long should we retain production API request logs?")
print(after_update.answer)
assert any(e.document.source_id == "comp-retention@v2#0" for e in after_update.citations)
assert any(e.document.source_id == "comp-retention@v1#0" for e in conflict_answer.citations)

Production API request logs must be deleted after 14 days under the current approved retention policy. This policy takes precedence over informal engineering notes. [comp-retention@v2#0]

Use load tests and CPU and memory measurements to decide whether API capacity needs scaling. Define a rollback threshold before changes. [tech-capacity@v1#0]

To troubleshoot API performance issues, inspect p95 latency, error rates and traces. Profile slow endpoints and database queries before changing capacity. [tech-performance@v1#0]

To deploy a microservice, run unit and integration tests, build a versioned container image, deploy to staging, then use a canary rollout with a rollback plan. [tech-deploy@v1#0]

When troubleshooting API performance, redact personal data and credentials from logs, use least-privilege access and never disable production audit logging. [comp-debug@v1#0]

Before deploying a microservice, complete dependency vulnerability scanning, secrets scanning and a security review. 

## 5. Simulated feedback changes retrieval ranking
Create two identical passages to isolate the effect from embedding similarity. A rating may
only target a cited current source, once per answer. Weights stay in [0.8, 1.2].
This is bounded reranking, not model training. Source authority remains the first conflict rule.

In [6]:
from rag_system.llm import MockLLM
from rag_system.orchestrator import Orchestrator
from rag_system.vector_store import VectorStoreManager

original = system.store.documents["tech-deploy"]
copy = original.model_copy(update={"id": "aaa-deploy"})
feedback_demo = Orchestrator(VectorStoreManager([copy, original], system.store.embedder), MockLLM())
question = "How do I deploy a microservice?"


def ranking():
    return [
        (e.document.id, e.feedback_weight)
        for e in feedback_demo.store.search(question, "technical", 2, 0)
    ]


before = ranking()
rated = feedback_demo.query(question)
weight = feedback_demo.submit_feedback(
    Feedback(request_id=rated.request_id, source_id=original.source_id, helpful=True)
)
after = ranking()
print("Controlled MOCK feedback experiment:", before, "→", after)
assert before[0][0] == "aaa-deploy" and after[0][0] == "tech-deploy"
assert weight == 1.05

Controlled MOCK feedback experiment: [('aaa-deploy', 1.0), ('tech-deploy', 1.0)] → [('tech-deploy', 1.05), ('aaa-deploy', 1.0)]


## 6. Unresolved conflicts and absent evidence
Equal-authority contradictions with similar scores are withheld. Unknown topics yield no evidence.
These controlled failure demonstrations use mock generation and the same actual embedding backend.

In [7]:
policy = system.store.documents["comp-retention"]
alternative = policy.model_copy(
    update={"id": "retention-alt", "value": "60", "text": policy.text.replace("14 days", "60 days")}
)
conflict_demo = Orchestrator(
    VectorStoreManager([policy, alternative], system.store.embedder), MockLLM()
)
unresolved = conflict_demo.query("How long should we retain production API request logs?")
print(unresolved.answer)
assert unresolved.conflicts and unresolved.conflicts[0].selected_source_id is None
unknown = system.query("Who won the World Cup?")
print(unknown.status, unknown.answer)
assert unknown.status == "no_evidence"

Insufficient evidence to answer this question.

Conflicting sources require clarification; disputed facts were withheld.
no_evidence Insufficient evidence to answer this question.


## 7. Partial failure is visible
A simulated compliance timeout must not turn into an apparently complete answer.

In [8]:
from rag_system.llm import LLMError

failure_demo = build_system(
    mode="mock", embedding_backend=os.getenv("EMBEDDING_BACKEND", "fastembed")
)
original_generate = failure_demo.llm.generate


def fail_compliance(stage, instructions, payload, schema):
    if stage == "domain" and payload["task"]["domain"] == "compliance":
        raise LLMError("Simulated compliance timeout")
    return original_generate(stage, instructions, payload, schema)


failure_demo.llm.generate = fail_compliance
partial = failure_demo.query(queries[0])
print(partial.status, partial.answer)
print([(r.task.domain, r.error) for r in partial.results])
assert partial.status == "partial"

partial To deploy a microservice, run unit and integration tests, build a versioned container image, deploy to staging, then use a canary rollout with a rollback plan. [tech-deploy@v1#0]

Implement a data processing workflow with schema validation, idempotent processing, retries and a dead-letter queue. Test it in staging before release. [tech-pipeline@v1#0]

Roll back a failed deployment to the previous image version. Verify health checks and error rates before resuming traffic. [tech-rollback@v1#0]

Use load tests and CPU and memory measurements to decide whether API capacity needs scaling. Define a rollback threshold before changes. [tech-capacity@v1#0]

This is a partial answer; consult the domain results for missing evidence.
[('technical', None), ('compliance', 'Simulated compliance timeout')]


## 8. Evaluate retrieval separately from generation
24 hand-authored development cases, including the required scenarios and unanswerable questions.
The report records raw retrieval recall separately from expected-source recall in the final answer.
A diagnostic pass is not proof of semantic support. Review claims against cited passages manually;
mock results do not measure the live LLM. The additional cases are not an independent holdout.

In [9]:
from rag_system.evaluation import evaluate

evaluation_system = build_system()
report = evaluate(evaluation_system)
print(json.dumps({k: v for k, v in report.items() if k != "rows"}, indent=2))
print("Cases needing review:")
for row in report["rows"]:
    if not row["diagnostic_pass"]:
        print(row)
assert report["cases"] == 24

{
  "mode": "mock",
  "embedding_backend": "fastembed:sentence-transformers/all-MiniLM-L6-v2",
  "cases": 24,
  "routing_accuracy": 1.0,
  "mean_retrieval_recall_at_dynamic_k": 1.0,
  "diagnostic_success_rate": 0.9583333333333334,
  "definition": "Diagnostic success checks routing, expected sources and abstention. It does not prove claim entailment or answer quality."
}
Cases needing review:
{'id': 'unknown-tech', 'group': 'additional', 'routing_correct': True, 'retrieval_recall_at_dynamic_k': None, 'expected_source_recall_in_answer': None, 'expected_status': False, 'status': 'answered', 'diagnostic_pass': False, 'latency_ms': 22.11, 'llm_calls': 3, 'retrieved': ['tech-performance', 'tech-rollback'], 'cited': ['tech-performance', 'tech-rollback']}


## 9. Performance monitoring and the OpenAPI boundary
Operational completion and explicit user feedback are separate metrics. HTTP 200 is not correctness.
This in-process HTTP check uses the same core. Run `uvicorn rag_system.api:app` for a standalone service.

In [10]:
from fastapi.testclient import TestClient

from rag_system.api import create_app

print(json.dumps(system.metrics(), indent=2))
with TestClient(create_app(system)) as client:
    assert client.get("/health").status_code == 200
    assert "/query" in client.get("/openapi.json").json()["paths"]
    api_answer = client.post("/query", json={"text": queries[0]})
    assert api_answer.status_code == 200
    print("API:", api_answer.json()["status"], api_answer.json()["mode"])
    assert client.post("/query", json={"text": " "}).status_code == 422

{
  "mode": "mock",
  "embedding_backend": "fastembed:sentence-transformers/all-MiniLM-L6-v2",
  "queries_total": 6,
  "outcomes_total": {
    "answered": 5,
    "no_evidence": 1
  },
  "query_completion_rate": 0.8333333333333334,
  "completion_definition": "All routed domains returned claims; not a factual correctness metric.",
  "feedback_count_in_window": 0,
  "feedback_success_rate_in_window": null,
  "retained_queries": 6,
  "agents": {
    "technical": {
      "runs_in_window": 5,
      "errors_in_window": 0,
      "mean_ms_in_window": 9.117025398882106
    },
    "business": {
      "runs_in_window": 1,
      "errors_in_window": 0,
      "mean_ms_in_window": 10.70958300260827
    },
    "compliance": {
      "runs_in_window": 5,
      "errors_in_window": 0,
      "mean_ms_in_window": 9.601100394502282
    }
  },
  "llm_calls_total": 22,
  "input_tokens_total": 0,
  "output_tokens_total": 0
}


API: answered mock


/private/tmp/rag-assessment-venv/lib/python3.13/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## Interview discussion
- Why exact in-memory search is sufficient for 18 short documents.
- What is simulated: mock generation/routing, annotated conflicts, feedback adaptation.
- What is real: embeddings (default), retrieval, citations, orchestration, API and updates.
- Why citation membership cannot prove entailment, and why similarity is not truth.
- When persistence, hybrid search, reranking and concurrency would be justified.
- State is process-local; notebook and API instances do not share updates.

The separate Systems Design assignment is intentionally not implemented here.